# NoPOS — Encoder Head Mask Experiments

Trains an encoder-decoder language model on WikiText-103 with different
per-head attention masks in the encoder (`C` = causal, `F` = future, `B` = bidirectional).

**Setup checklist (do this before running):**
1. Runtime → Change runtime type → GPU → **H100** (or A100)
2. Run cells **1 → 2 → 3 → 4** once per session to set up the environment
3. Edit **Cell 5** to choose your experiment (`COND`, `SPEC`, etc.)
4. Run **Cell 6** to train; re-run it to resume after a session restart

Checkpoints and logs are saved to Google Drive and survive session restarts.

In [4]:
# Cell 1 — Verify GPU
!nvidia-smi
import torch
print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sun May 24 03:27:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
%%bash
# Cell 2 — Set up repo & install
# Pulls the repo from Google Drive (nopos_experiments/nopos.zip) if present,
# otherwise falls back to cloning from GitHub.
set -e

DRIVE_ZIP="/content/drive/MyDrive/nopos_experiments/nopos.zip"

if [ ! -d /content/nopos ]; then
    if [ -f "$DRIVE_ZIP" ]; then
        echo "Extracting repo from Drive ($DRIVE_ZIP) ..."
        unzip -q "$DRIVE_ZIP" -d /content/
        echo "Extracted."
    else
        echo "No Drive zip found — cloning Anxinal/futureMask (branch: master) ..."
        git clone --branch master https://github.com/Anxinal/futureMask.git /content/nopos
    fi
else
    echo "Repo already present at /content/nopos."
fi

cd /content/nopos

echo "Pinning pip to <24.1 ..."
pip install -q "pip<24.1"

echo "Installing Python dependencies ..."
pip install -q "hydra-core>=1.0.7,<1.1" "omegaconf<2.1" \
    regex sacrebleu tqdm bitarray cffi cython

echo "Installing fairseq (editable) ..."
pip install -q -e .

python -c "import fairseq; print('fairseq OK:', fairseq.__version__)"
echo "--- Setup complete ---"

In [ ]:
# Cell 3 — Mount Google Drive
# Checkpoints and logs are stored here so they survive session restarts.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nopos_experiments'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.environ['DRIVE_DIR'] = DRIVE_DIR   # make available to %%bash cells
print(f"Drive mounted. All outputs → {DRIVE_DIR}")

In [ ]:
%%bash
# Cell 4 — Download & preprocess WikiText-103
#
# Logic (checked in order):
#   1. data-bin already in /content  → nothing to do
#   2. data-bin cached on Drive      → copy from Drive (~30 s)
#   3. neither                       → download + preprocess (~5 min), then cache to Drive
set -e

DATABIN="/content/nopos/data-bin/wikitext-103"
DRIVE_DATABIN="${DRIVE_DIR}/data-bin/wikitext-103"

if [ -d "$DATABIN" ] && [ -n "$(ls -A $DATABIN 2>/dev/null)" ]; then
    echo "Preprocessed data already in /content — nothing to do."

elif [ -d "$DRIVE_DATABIN" ]; then
    echo "Restoring preprocessed data from Drive ..."
    mkdir -p "$(dirname $DATABIN)"
    cp -r "$DRIVE_DATABIN" "$DATABIN"
    echo "Restored."

else
    echo "Downloading WikiText-103 (~180 MB) ..."
    mkdir -p /content/wt103-raw
    wget -q -O /content/wt103-raw/wt103.zip \
        https://dl.fbaipublicfiles.com/fairseq/data/wikitext-103-v1.zip
    unzip -q /content/wt103-raw/wt103.zip -d /content/wt103-raw/

    echo "Preprocessing (dictionary build + binarise) ..."
    cd /content/nopos
    python fairseq_cli/preprocess.py \
        --only-source \
        --trainpref /content/wt103-raw/wikitext-103/wiki.train.tokens \
        --validpref /content/wt103-raw/wikitext-103/wiki.valid.tokens \
        --testpref  /content/wt103-raw/wikitext-103/wiki.test.tokens \
        --destdir   "$DATABIN" \
        --workers   4

    echo "Caching to Drive for future sessions ..."
    mkdir -p "$(dirname $DRIVE_DATABIN)"
    cp -r "$DATABIN" "$(dirname $DRIVE_DATABIN)/"
    echo "Done."
fi

echo ""
echo "Contents of data-bin:"
ls /content/nopos/data-bin/wikitext-103/

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           Cell 5 — CONFIGURE YOUR EXPERIMENT               ║
# ║   Edit the values in this cell, then run Cell 6 to train   ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Experiment identity ─────────────────────────────────────────
COND = "CFBBBBBB"           # Name used for log/checkpoint files
SPEC = "C,F,B,B,B,B,B,B"   # Encoder head mask per head (8 heads total)
                             #   C = causal (attend to past only)
                             #   F = future (attend to future only)
                             #   B = bidirectional (unrestricted)

# ── Batch & compute ─────────────────────────────────────────────
# Local baseline used max_tokens=2048 on RTX 4060.
# H100 can handle much larger batches — increase MAX_TOKENS for
# faster throughput. Effective batch = MAX_TOKENS * UPDATE_FREQ.
# To keep the same update dynamics as local, keep MAX_TOKENS=2048.
MAX_TOKENS    = 2048    # tokens per GPU step (increase on H100, e.g. 8192)
UPDATE_FREQ   = 1       # gradient accumulation steps
MAX_UPDATES   = 100000  # total gradient updates
VALIDATE_EVERY = 2000   # run validation every N updates
SAVE_EVERY     = 8000   # checkpoint every N updates

# ── Derived paths (no need to edit) ─────────────────────────────
import os
DRIVE_DIR = os.environ.get('DRIVE_DIR', '/content/drive/MyDrive/nopos_experiments')
SAVE_DIR  = f'{DRIVE_DIR}/checkpoints/{COND}'
LOG_FILE  = f'{DRIVE_DIR}/{COND}.log'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Condition    : {COND}")
print(f"Head mask    : {SPEC}")
print(f"Max tokens   : {MAX_TOKENS}  (update_freq={UPDATE_FREQ})")
print(f"Max updates  : {MAX_UPDATES}")
print(f"Validate every {VALIDATE_EVERY} | Save every {SAVE_EVERY}")
print(f"Save dir     : {SAVE_DIR}")
print(f"Log file     : {LOG_FILE}")

ckpt = os.path.join(SAVE_DIR, 'checkpoint_last.pt')
if os.path.exists(ckpt):
    print(f"\nFound existing checkpoint — Cell 6 will RESUME from it.")
else:
    print("\nNo existing checkpoint — Cell 6 will start fresh.")

In [ ]:
# Cell 6 — Run training
# Re-run this cell after a session restart to resume from the Drive checkpoint.
import subprocess, sys, json, re, os
from datetime import datetime

DATABIN = '/content/nopos/data-bin/wikitext-103'

cmd = [
    sys.executable, '/content/nopos/fairseq_cli/train.py', DATABIN,
    '--task',                         'encoder_decoder_language_modeling',
    '--sample-break-mode',            'none',
    '--tokens-per-sample',            '512',
    '--encoder-prefix-fraction',      '0.5',
    '--arch',                         'transformer',
    '--encoder-layers',               '8',
    '--decoder-layers',               '8',
    '--encoder-attention-heads',      '8',
    '--decoder-attention-heads',      '8',
    '--encoder-embed-dim',            '512',
    '--decoder-embed-dim',            '512',
    '--encoder-ffn-embed-dim',        '512',
    '--decoder-ffn-embed-dim',        '512',
    '--share-all-embeddings',
    '--no-token-positional-embeddings',
    '--encoder-head-mask-spec',       SPEC,
    '--dropout',                      '0.1',
    '--attention-dropout',            '0.1',
    '--optimizer',                    'adam',
    '--adam-betas',                   '(0.9, 0.98)',
    '--weight-decay',                 '0.01',
    '--clip-norm',                    '1.0',
    '--lr',                           '5e-4',
    '--lr-scheduler',                 'inverse_sqrt',
    '--warmup-updates',               '500',
    '--criterion',                    'cross_entropy',
    '--max-tokens',                   str(MAX_TOKENS),
    '--update-freq',                  str(UPDATE_FREQ),
    '--max-update',                   str(MAX_UPDATES),
    '--skip-invalid-size-inputs-valid-test',
    '--required-batch-size-multiple', '1',
    '--validate-interval-updates',    str(VALIDATE_EVERY),
    '--save-interval-updates',        str(SAVE_EVERY),
    '--keep-interval-updates',        '1',
    '--keep-best-checkpoints',        '1',
    '--no-epoch-checkpoints',
    '--log-interval',                 '100',
    '--log-format',                   'json',
    '--fp16',
    '--num-workers',                  '4',
    '--seed',                         '1',
    '--save-dir',                     SAVE_DIR,
]

print('=' * 65)
print(f'  Condition   : {COND}')
print(f'  Head mask   : {SPEC}')
print(f'  Batch       : max_tokens={MAX_TOKENS} x update_freq={UPDATE_FREQ}')
print(f'  Max updates : {MAX_UPDATES}')
print(f'  Save dir    : {SAVE_DIR}')
print(f'  Log file    : {LOG_FILE}')
ckpt = os.path.join(SAVE_DIR, 'checkpoint_last.pt')
print(f'  Resuming    : {os.path.exists(ckpt)}')
print('=' * 65 + '\n')

best_ppl, last_train_upd = float('inf'), -1

def on_line(line, log_fh):
    global best_ppl, last_train_upd
    log_fh.write(line)
    log_fh.flush()
    m = re.search(r'\| (train|valid) \| (\{.+\})\s*$', line)
    if not m:
        return
    kind = m.group(1)
    try:
        row = json.loads(m.group(2))
    except json.JSONDecodeError:
        return
    ts = datetime.now().strftime('%H:%M:%S')
    if kind == 'train':
        upd = int(row.get('train_num_updates', 0))
        if upd - last_train_upd >= 1000:
            wps = float(row.get('train_wps', 0))
            print(f'[{ts}] train | upd={upd:>7} | loss={row.get("train_loss"):>6} | '
                  f'ppl={row.get("train_ppl"):>8} | wps={wps:>8.0f}')
            last_train_upd = upd
    elif kind == 'valid':
        upd = int(row.get('valid_num_updates', 0))
        ppl = float(row.get('valid_ppl', 'inf'))
        flag = '  ★ BEST' if ppl < best_ppl else ''
        best_ppl = min(best_ppl, ppl)
        print(f'[{ts}] VALID | upd={upd:>7} | loss={row.get("valid_loss"):>6} | ppl={ppl:>8.2f}{flag}')

with open(LOG_FILE, 'a') as log_fh:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd='/content/nopos'
    )
    try:
        for line in proc.stdout:
            on_line(line, log_fh)
    except KeyboardInterrupt:
        proc.terminate()
        print(f'\nInterrupted. Best PPL so far: {best_ppl:.2f}')
        print(f'Checkpoint saved to {SAVE_DIR}')
    proc.wait()

if best_ppl < float('inf'):
    print(f'\nDone. Best valid PPL = {best_ppl:.2f}')
print(f'Log → {LOG_FILE}')

In [ ]:
# Cell 7 — View validation results from log
import json, re, os

log_to_show = LOG_FILE  # or set to any path, e.g. f'{DRIVE_DIR}/CCCCBBBB.log'

valid_rows, best_ppl = [], float('inf')
with open(log_to_show) as f:
    for line in f:
        m = re.search(r'\| valid \| (\{.+\})\s*$', line)
        if m:
            try:
                valid_rows.append(json.loads(m.group(1)))
            except json.JSONDecodeError:
                pass

if not valid_rows:
    print('No validation entries found in log yet.')
else:
    cond_name = os.path.basename(log_to_show).replace('.log', '')
    print(f'Condition : {cond_name}')
    print(f'{"Updates":>10}  {"Valid Loss":>12}  {"Valid PPL":>12}')
    print('-' * 42)
    for r in valid_rows:
        upd  = int(r.get('valid_num_updates', 0))
        loss = float(r.get('valid_loss', 0))
        ppl  = float(r.get('valid_ppl', 0))
        marker = '  <- BEST' if ppl < best_ppl else ''
        best_ppl = min(best_ppl, ppl)
        print(f'{upd:>10}  {loss:>12.4f}  {ppl:>12.2f}{marker}')
    print(f'\nBest valid PPL : {best_ppl:.2f}')

In [ ]:
# Cell 8 — (Optional) Run multiple conditions sequentially
# Useful for overnight runs. Each condition resumes from Drive if interrupted.
import subprocess, sys, re, json, os
from datetime import datetime

CONDITIONS = [
    # (COND,          SPEC)
    ('BBBBBBBB',  'B,B,B,B,B,B,B,B'),
    ('CFBBBBBB',  'C,F,B,B,B,B,B,B'),
    ('CCFFBBBB',  'C,C,F,F,B,B,B,B'),
    ('CCCCBBBB',  'C,C,C,C,B,B,B,B'),
    ('CCCCFFFF',  'C,C,C,C,F,F,F,F'),
]

DATABIN        = '/content/nopos/data-bin/wikitext-103'
DRIVE_DIR      = os.environ.get('DRIVE_DIR', '/content/drive/MyDrive/nopos_experiments')
MAX_TOKENS_ALL = 2048    # change to 8192 to use H100's full capacity
UPDATE_FREQ_ALL = 1
MAX_UPDATES_ALL = 100000

def build_cmd(cond, spec, save_dir):
    return [
        sys.executable, '/content/nopos/fairseq_cli/train.py', DATABIN,
        '--task',                         'encoder_decoder_language_modeling',
        '--sample-break-mode',            'none',
        '--tokens-per-sample',            '512',
        '--encoder-prefix-fraction',      '0.5',
        '--arch',                         'transformer',
        '--encoder-layers',               '8',
        '--decoder-layers',               '8',
        '--encoder-attention-heads',      '8',
        '--decoder-attention-heads',      '8',
        '--encoder-embed-dim',            '512',
        '--decoder-embed-dim',            '512',
        '--encoder-ffn-embed-dim',        '512',
        '--decoder-ffn-embed-dim',        '512',
        '--share-all-embeddings',
        '--no-token-positional-embeddings',
        '--encoder-head-mask-spec',       spec,
        '--dropout',                      '0.1',
        '--attention-dropout',            '0.1',
        '--optimizer',                    'adam',
        '--adam-betas',                   '(0.9, 0.98)',
        '--weight-decay',                 '0.01',
        '--clip-norm',                    '1.0',
        '--lr',                           '5e-4',
        '--lr-scheduler',                 'inverse_sqrt',
        '--warmup-updates',               '500',
        '--criterion',                    'cross_entropy',
        '--max-tokens',                   str(MAX_TOKENS_ALL),
        '--update-freq',                  str(UPDATE_FREQ_ALL),
        '--max-update',                   str(MAX_UPDATES_ALL),
        '--skip-invalid-size-inputs-valid-test',
        '--required-batch-size-multiple', '1',
        '--validate-interval-updates',    '2000',
        '--save-interval-updates',        '8000',
        '--keep-interval-updates',        '1',
        '--keep-best-checkpoints',        '1',
        '--no-epoch-checkpoints',
        '--log-interval',                 '100',
        '--log-format',                   'json',
        '--fp16',
        '--num-workers',                  '4',
        '--seed',                         '1',
        '--save-dir',                     save_dir,
    ]

summary = []

for cond, spec in CONDITIONS:
    save_dir = f'{DRIVE_DIR}/checkpoints/{cond}'
    log_file = f'{DRIVE_DIR}/{cond}.log'
    os.makedirs(save_dir, exist_ok=True)

    ckpt = os.path.join(save_dir, 'checkpoint_last.pt')
    ts0 = datetime.now().strftime('%H:%M:%S')
    print(f'\n[{ts0}] Starting {cond} (spec={spec}, resume={os.path.exists(ckpt)})')

    best_ppl, last_upd = float('inf'), -1

    with open(log_file, 'a') as log_fh:
        proc = subprocess.Popen(
            build_cmd(cond, spec, save_dir),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, cwd='/content/nopos'
        )
        for line in proc.stdout:
            log_fh.write(line)
            log_fh.flush()
            m = re.search(r'\| (train|valid) \| (\{.+\})\s*$', line)
            if not m:
                continue
            kind = m.group(1)
            try:
                row = json.loads(m.group(2))
            except json.JSONDecodeError:
                continue
            ts = datetime.now().strftime('%H:%M:%S')
            if kind == 'train':
                upd = int(row.get('train_num_updates', 0))
                if upd - last_upd >= 5000:
                    print(f'  [{ts}] train upd={upd} ppl={row.get("train_ppl")}')
                    last_upd = upd
            elif kind == 'valid':
                upd = int(row.get('valid_num_updates', 0))
                ppl = float(row.get('valid_ppl', 'inf'))
                flag = ' ★' if ppl < best_ppl else ''
                best_ppl = min(best_ppl, ppl)
                print(f'  [{ts}] VALID upd={upd} ppl={ppl:.2f}{flag}')
        proc.wait()

    summary.append((cond, spec, best_ppl))
    print(f'  Finished {cond} — best PPL = {best_ppl:.2f}')

print('\n' + '=' * 55)
print(f'{"Condition":<14}  {"Spec":<22}  {"Best Valid PPL":>14}')
print('-' * 55)
for cond, spec, ppl in sorted(summary, key=lambda x: x[2]):
    print(f'{cond:<14}  {spec:<22}  {ppl:>14.2f}')
print('=' * 55)